# Signals on the Sphere — Companion Notebook

**6.7970/8.750 Symmetry and its Application to Machine Learning**

This notebook follows the Signals on the Sphere exercise section by section.
Use it to **prototype your code** and **test your implementations**
against the course library before submitting on the website.

Each section includes small tests you can use to check your work.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atomicarchitects/symm4ml-colabs/blob/main/spherical_companion.ipynb)

## Setup

In [ ]:
%%capture
!pip install https://symm4ml.mit.edu/_static/symm4ml_s26/symm4ml/symm4ml_latest.zip

In [ ]:
from typing import List

import numpy as np
import scipy.linalg

from symm4ml import lie, so3

### Reference data

These arrays are used throughout the exercise for testing.

In [ ]:
# SO(3) L=1 generators
so3_gens = np.array(
    [
        [[0.0, 0.0, 0.0], [0.0, 0.0, -1.0], [0.0, 1.0, 0.0]],
        [[0.0, 0.0, 1.0], [0.0, 0.0, 0.0], [-1.0, 0.0, 0.0]],
        [[0.0, -1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
    ]
)

# SO(3) irrep generators L=0..3 (used by all problems)
Xs = lie.infer_irreps_from_tensor_products(so3_gens, 4)

# Random spherical harmonic coefficients (used by convolution test)
sh_rand = [
    np.array([0.646]),
    np.array([0.892, 0.33, 0.406]),
    np.array([0.822, 0.204, 0.7, 0.118, 0.857]),
    np.array([0.397, 0.374, 0.269, 0.291, 0.79, 0.71, 0.169]),
]

print(f"Xs: {len(Xs)} irreps, dimensions: {[X.shape[-1] for X in Xs]}")
print(f"sh_rand: {len(sh_rand)} levels, dimensions: {[s.shape[0] for s in sh_rand]}")

---
## Section 1: Understanding Signals on Spheres

### Conceptual Questions on Spherical Harmonics

The spherical harmonics $Y^L_m(\hat{x})$ form a complete orthonormal basis for square-integrable functions on the sphere $S^2$. They are intimately connected to the irreps of $SO(3)$: the $2L+1$ spherical harmonics of a given $L$ transform as the $L$-th irrep under rotations.

Any smooth scalar function on the sphere can be expanded as:
$$f(\hat{x}) = \sum_{L=0}^{\infty} \sum_{m=-L}^{L} \hat{f}_m^{\,L} \, Y^L_m(\hat{x})$$

where $\hat{f}_m^{\,L}$ are the spherical harmonic coefficients.

In [ ]:
# YOUR ANSWER HERE:
# sh_facts: Which statements about spherical harmonics are true? (select all that apply)
# Options:
# 1. The integral of two inequivalent spherical harmonics over S^2 is zero.
# 2. The pointwise product of two spherical harmonics (single L1,m1 and L2,m2) produces only a single spherical harmonic (single L3,m3).
# 3. Spherical harmonics are the simplest polynomials of (x,y,z) that transform as irreps of SO(3).
# 4. The spherical harmonics span the irreps of O(3).
# 5. Any (finite, smooth) scalar function on the sphere can be represented as an expansion of spherical harmonics.
# 6. To represent pseudoscalar functions on the sphere, we use spherical harmonics of opposite parity.

In [ ]:
# YOUR ANSWER HERE:
# sh_dimension: For a given L, how many spherical harmonics are there?
# Options:
# 1. (L+1)^2
# 2. L
# 3. 2L+1

In [ ]:
# YOUR ANSWER HERE:
# sh_irreps: Which of the following irreps do spherical harmonics transform as? (select all that are true)
# Options:
# 1. (L=0, p=even)
# 2. (L=0, p=odd)
# 3. (L=1, p=even)
# 4. (L=1, p=odd)
# 5. (L=2, p=even)
# 6. (L=2, p=odd)

---
### Projection of Dirac delta onto Spherical Harmonics

The Dirac delta function $f(x) = \delta(x - x_0)$ is a discrete function, equal to 0 everywhere except at $x_0$. Dirac delta functions have the property: $\int_x f(x)g(x)dx = g(x_0)$, regardless of $g$.

Use this property to compute the first $L$ spherical harmonic coefficients of $f(x) = \delta(x - x_0)$ where $\|x_0\| = 1$. ($L$ is implicitly defined by the number of elements in `Xs`.)

**Hint:** the solution should be roughly one line. Recall that the spherical harmonic coefficients can be written as a convolution (integral).

In [ ]:
def project_delta_function(Xs: List[np.ndarray], x0: np.ndarray) -> List[np.ndarray]:
    r"""Project a delta function onto the first L spherical harmonics.
    Input:
        Xs: list of irreps of SO(3), from l=0, 1, 2, ... to l=L (each evaluated at the generators: therefore, an np.array [3, 2*l+1, 2*l+1])
        x0: a unit vector in R^3, i.e. an np.array of size [3], such that
            the delta function to project is equal to f(x) = delta(x-x0)
    Output:
        sh: list of the spherical harmonic coefficients of f(x), from l=0, 1, 2, ... to l=L (each an np.array [2*l+1])
    Note:
        As normalisation, we normalise the spherical harmonics to have unit norm. (In other words, each output array in the list sh should have unit norm.)
    """
    # YOUR CODE HERE
    pass

In [ ]:
# No small tests in so3.py for project_delta_function.
# Supplementary comparison with course (not a course small test):
result = project_delta_function(Xs, np.array([0.0, 1.0, 0.0]))
ref = so3.project_delta_function(Xs, np.array([0.0, 1.0, 0.0]))
for l in range(len(Xs)):
    np.testing.assert_allclose(result[l], ref[l], atol=1e-7)

result2 = project_delta_function(Xs, np.array([0.70710678, 0.70710678, 0.0]))
ref2 = so3.project_delta_function(Xs, np.array([0.70710678, 0.70710678, 0.0]))
for l in range(len(Xs)):
    np.testing.assert_allclose(result2[l], ref2[l], atol=1e-7)
print("project_delta_function comparison passed!")

---
### Spherical harmonic projection of function at points on sphere

Suppose you wish to compute the spherical harmonics of a function $f: S^2 \rightarrow \mathbb{C}$, but you only know its values at a discrete set of points: $f(\ell_i) = v_i$ for $m$ locations $\{\ell_i\}_{i=1}^m$. One option is to interpret $f$ as a discrete sum of weighted Dirac delta functions:
$$f(x) = \sum_{i=1}^m v_i \, \delta(x - \ell_i)$$

Using this interpretation and the previous problem, compute the spherical harmonic coefficients $\hat{f}_m^{\,L}$ of $f$.

**Hint:** Spherical harmonic coefficients are linear functions.

In [ ]:
def project_discretized_function(
    Xs: List[np.ndarray], locations: List[np.ndarray], values: np.ndarray
):
    """Project a discretized function onto the first L spherical harmonics.
    Input:
        Xs: list of irreps of SO(3), from l=0, 1, 2, ... to l=L (each evaluated at the generators: therefore, an np.array [3, 2*l+1, 2*l+1])
        locations: a list of N unit vectors in R^3, i.e. each an np.array of size [3]
        values: an np.array of size [N], such that
            the input function f(x) = values[i] if x=locations[i], and f(x) is equal to 0 otherwise
    Output:
        sh: list of the spherical harmonic coefficients of f(x), from l=0, 1, 2, ... to l=L (each an np.array [2*l+1])
    Note:
        As normalisation, we normalise the spherical harmonics to have unit norm. (In other words, each output array in the list sh should have unit norm.)
    """
    # YOUR CODE HERE
    pass

In [ ]:
# No small tests in so3.py for project_discretized_function.
# Supplementary comparison with course (not a course small test):
locs1 = [np.array([0.70710678, 0.70710678, 0.0]), np.array([1, 0, 0])]
vals1 = np.array([3.2, 4.1])
result = project_discretized_function(Xs, locs1, vals1)
ref = so3.project_discretized_function(Xs, locs1, vals1)
for l in range(len(Xs)):
    np.testing.assert_allclose(result[l], ref[l], atol=1e-7)

locs2 = [np.array([0.0, 1.0, 0.0]), np.array([1, 0, 0])]
vals2 = np.array([2.5, 1.2])
result2 = project_discretized_function(Xs, locs2, vals2)
ref2 = so3.project_discretized_function(Xs, locs2, vals2)
for l in range(len(Xs)):
    np.testing.assert_allclose(result2[l], ref2[l], atol=1e-7)
print("project_discretized_function comparison passed!")

---
### Rotate function on sphere

Consider a function $f: S^2 \rightarrow \mathbb{C}$. Given a rotation matrix $R$ (satisfying $R^T R = I$ and $\det R = 1$), the rotated function $f'$ is defined by $f'(x) = f(R^T x)$.

There is a simple relationship between the spherical harmonic coefficients of $f$ and those of $f'$, expressed in terms of the Wigner-D matrices (the irreps of $SO(3)$). The Wigner-D matrix $D^L(R)$ for a rotation $R = e^{\theta \hat{n} \cdot \mathbf{L}}$ is:

$$D^L(R) = e^{\theta \hat{n} \cdot \mathbf{X}^L}$$

where $\mathbf{X}^L$ are the $L$-th irrep generators. The rotated coefficients are then $\hat{f}'^{\,L} = D^L(R) \, \hat{f}^{\,L}$.

You can use `scipy.linalg.expm` to compute the matrix exponential.

In [ ]:
def rotate_function(
    Xs: List[np.ndarray], sh: List[np.ndarray], axis: np.ndarray, angle: float
) -> List[np.ndarray]:
    """Rotate a function f(x), given in terms of its spherical harmonic coefficients,
    Input:
        Xs: list of irreps of SO(3), from l=0, 1, 2, ... to l=L (each evaluated at the generators: therefore, an np.array [3, 2*l+1, 2*l+1])
        sh: list of the spherical harmonic coefficients of f(x), from l=0, 1, 2, ... to l=L (each an np.array [2*l+1])
        axis: a unit vector representing the axis about which to rotate f(x), np.array of shape [3]
        angle: the angle (in radians) by which to rotate f(x) around axis
    Output:
        sh_rotated: list of the spherical harmonic coefficients of f(x) AFTER rotation, from l=0, 1, 2, ... to l=L (each an np.array [2*l+1])
    """
    # YOUR CODE HERE
    pass

In [ ]:
# No small tests in so3.py for rotate_function.
# Supplementary comparison with course (not a course small test):
sh_delta = so3.project_delta_function(Xs, np.array([0.70710678, 0.70710678, 0.0]))

result = rotate_function(
    Xs=Xs, sh=sh_delta,
    axis=np.array([0.70710678, 0.70710678, 0.0]),
    angle=0.3 * np.pi,
)
ref = so3.rotate_function(
    Xs=Xs, sh=sh_delta,
    axis=np.array([0.70710678, 0.70710678, 0.0]),
    angle=0.3 * np.pi,
)
for l in range(len(Xs)):
    np.testing.assert_allclose(result[l], ref[l], atol=1e-7)

# Test with a discretized function
sh_disc = so3.project_discretized_function(
    Xs,
    [np.array([0.70710678, 0.70710678, 0.0]), np.array([1, 0, 0])],
    np.array([3.2, 4.1]),
)
result2 = rotate_function(Xs=Xs, sh=sh_disc, axis=np.array([0, 1, 0]), angle=0.5 * np.pi)
ref2 = so3.rotate_function(Xs=Xs, sh=sh_disc, axis=np.array([0, 1, 0]), angle=0.5 * np.pi)
for l in range(len(Xs)):
    np.testing.assert_allclose(result2[l], ref2[l], atol=1e-7)
print("rotate_function comparison passed!")

#### Verify equivariance: rotating then projecting = projecting then rotating

A key property is that projecting a function onto spherical harmonics and then rotating the coefficients should give the same result as rotating the function first and then projecting. You can verify this with your implementations:

In [ ]:
# Equivariance check (uses course library implementations):
# "project then rotate" vs "rotate then project"
x0 = np.array([1.0, 0.0, 0.0])
axis = np.array([0.0, 0.0, 1.0])
angle = np.pi / 3

# Path 1: project delta at x0, then rotate coefficients
sh_proj = so3.project_delta_function(Xs, x0)
sh_proj_rot = so3.rotate_function(Xs, sh_proj, axis, angle)

# Path 2: rotate x0, then project delta at rotated point
R = so3.axis_angle_to_matrix(axis, angle)
x0_rot = R @ x0
sh_rot_proj = so3.project_delta_function(Xs, x0_rot)

for l in range(len(Xs)):
    np.testing.assert_allclose(sh_proj_rot[l], sh_rot_proj[l], atol=1e-7)
print("Equivariance verified: rotate-then-project == project-then-rotate")

---
## Section 2: Spherical Convolution

Let $f$ and $g$ be spherical functions: $f, g: S^2 \rightarrow \mathbb{C}$. The spherical convolution is $c: SO(3) \rightarrow \mathbb{C}$, where $c(R) = \int_{x \in S^2} f(R^T x) g(x) dx$.

By the Peter-Weyl theorem, $c$ can be expressed in terms of Wigner-D coefficients $c_L \in \mathbb{C}^{(2L+1) \times (2L+1)}$.

The **convolution theorem** tells us that the Wigner-D coefficients of $c$ are related simply to the spherical harmonic coefficients of $f$ and $g$. This is analogous to how Fourier coefficients behave under convolution — except here each "coefficient" is a vector (the spherical harmonics for a given $L$), and the "multiplication" in Fourier space produces a matrix.

This Fourier-space view of convolution is exactly how [Spherical CNNs](https://arxiv.org/pdf/1801.10130.pdf) implement their linear layers.

In [ ]:
def convolve_spherical_functions(sh1: List[np.ndarray], sh2: List[np.ndarray]):
    """Convolve two spherical functions, each given in terms of their spherical harmonic coefficients
    Input:
        sh1: list of the spherical harmonic coefficients of f(x), from l=0, 1, 2, ... to l=L (each an np.array [2*l+1])
        sh2: list of the spherical harmonic coefficients of h(x), from l=0, 1, 2, ... to l=L (each an np.array [2*l+1])
    Output:
        convolved_fxn: convolution of f(x) and h(x), expressed in the basis of irreps (Wigner-D matrices) as a list indexed by l=0, 1, 2, ... to l=L (each element an np.array [2*l+1, 2*l+1])
    """
    # YOUR CODE HERE
    pass

In [ ]:
# No small tests in so3.py for convolve_spherical_functions.
# Supplementary comparison with course (not a course small test):
sh_delta = so3.project_delta_function(Xs, np.array([0.70710678, 0.70710678, 0.0]))
sh_disc = so3.project_discretized_function(
    Xs,
    [np.array([0.70710678, 0.70710678, 0.0]), np.array([1, 0, 0])],
    np.array([3.2, 4.1]),
)

result = convolve_spherical_functions(sh_delta, sh_disc)
ref = so3.convolve_spherical_functions(sh_delta, sh_disc)
for l in range(len(Xs)):
    np.testing.assert_allclose(result[l], ref[l], atol=1e-7)

result2 = convolve_spherical_functions(sh_rand, sh_rand)
ref2 = so3.convolve_spherical_functions(sh_rand, sh_rand)
for l in range(len(Xs)):
    np.testing.assert_allclose(result2[l], ref2[l], atol=1e-7)
print("convolve_spherical_functions comparison passed!")

---
## Explore Further

In [ ]:
# Try projecting different functions onto spherical harmonics,
# rotating them, and convolving them!